# Full pipeline

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pedroliman/heormodel/blob/main/docs/_notebooks/full-pipeline.ipynb)

In [ ]:
# Install heormodel from PyPI.
%pip install -q heormodel

This tutorial shows how to run the standard probabilistic sensitivity analysis workflow end to end: specify parameters, sample draws, evaluate a model with `run_psa`, then pass the outcomes to cost-effectiveness and value-of-information analysis. [Bring your own outputs](https://pedroliman.github.io/heormodel/tutorials/byo-outputs.html) covers the analysis functions in more detail; here the focus is the run itself.

## Specifying and sampling parameters

Distributions are specified directly or from published mean and standard error. `ParameterSet.sample` returns a draw matrix, one row per iteration; a Spearman rank correlation can be requested per parameter pair.

In [ ]:
from heormodel.params import Beta, Gamma, Normal, ParameterSet
from heormodel.run import SeedManager

params = ParameterSet(
    {
        "p_event": Beta.from_mean_se(0.20, 0.03),
        "rr_treat": Normal(0.70, 0.10),
        "c_event": Gamma.from_mean_se(25_000, 3_000),
        "c_treat": Gamma.from_mean_se(4_000, 500),
        "u_event": Beta.from_mean_se(0.65, 0.05),
    }
)
seeds = SeedManager(42)
draws = params.sample(4_000, seed=seeds.generator())
draws.head(3).round(3)

Each row is one parameter draw. The model is evaluated once per row.

## Writing the model as a function

A model is a function that takes the draw matrix and returns `Outcomes`. The one below is a two-intervention decision tree evaluated over one year. `Outcomes.from_wide` builds that result from per-intervention cost and effect columns.

In [ ]:
import pandas as pd
from heormodel.models import Outcomes

def decision_tree(d: pd.DataFrame) -> Outcomes:
    p_treated = d["p_event"] * d["rr_treat"]
    costs = pd.DataFrame({
        "No treatment": d["p_event"] * d["c_event"],
        "Treatment": d["c_treat"] + p_treated * d["c_event"],
    })
    effects = pd.DataFrame({
        "No treatment": 1 - d["p_event"] * (1 - d["u_event"]),
        "Treatment": 1 - p_treated * (1 - d["u_event"]),
    })
    return Outcomes.from_wide(costs, effects)

## Running the probabilistic sensitivity analysis

`run_psa` evaluates the model on every draw and keeps each outcome matched to the parameters that produced it, the matching that value-of-information analysis relies on. It runs in parallel by default; this run is small, so `sequential=True` runs it without that overhead.

In [ ]:
from heormodel.run import run_psa

outcomes = run_psa(decision_tree, draws, sequential=True).outcomes
outcomes.summary().round(3)

## Analyzing cost-effectiveness and value of information

Nothing from here on is specific to a decision tree: the same `icer_table`, `evpi`, and `evppi_ranking` calls apply to the `Outcomes` any model produces.

In [ ]:
import numpy as np
from heormodel.cea import icer_table
from heormodel.voi import evpi, evppi_ranking

icer_table(outcomes).round(3)

Treatment is the more expensive, more effective intervention. Whether its incremental cost-effectiveness ratio stays below a given willingness-to-pay threshold, and how much resolving that uncertainty would be worth, are the questions the expected value of perfect information (EVPI) and the per-parameter ranking answer next.

In [ ]:
WTP = 50_000.0
print(f"EVPI at WTP {WTP:,.0f}: {evpi(outcomes, WTP):,.2f}")
evppi_ranking(outcomes, draws, WTP).round(2)

## Checking convergence with running means

`running_means` tracks the running mean of an outcome column per intervention as iterations accumulate; flat traces at the right edge suggest the iteration count is adequate for the means (value-of-information estimates typically need more).

In [ ]:
from heormodel.run import running_means

running_means(outcomes).plot(xlabel="iterations", ylabel="mean cost");

The [engines](https://pedroliman.github.io/heormodel/concepts/engines.html) concept page describes the model types `run_psa` can run. Next: the [value of information](https://pedroliman.github.io/heormodel/tutorials/voi.html) tutorial takes outcomes like these through the expected value of perfect, partial perfect, and sample information, checking each against a closed form.